# Building Thermal Simulation with Thermostat Setpoint Control

*`smart_control/simulator/building_setpoint_control_demo_ajk.ipynb`*

This notebook demonstrates how the **SBSim** building simulator models HVAC
thermostat setpoint control for a two-zone office building (`room_1`,
`room_2`). It runs a winter heating and a summer cooling scenario, each for
3 days (see `N_DAYS` in Section 6), and produces 12-row debug
visualisations of zone temperatures, HVAC component states (damper, reheat
valve, AHU, boiler), and energy rates, plus a floor-plan spatial heatmap.

## What you will see
1. A hard-coded 2-zone floor plan (23 × 12 control-volume grid)
2. Realistic material properties and HVAC devices, plus an illustrative
   occupancy schedule (Section 5 — **not** coupled into the thermal solver)
3. A **winter heating** scenario – the building starts cold (275 K) and the
   boiler + VAVs warm it up to the comfort band
4. A **summer cooling** scenario – the building starts hot (310 K) and the
   air handler attempts to cool it back into the comfort band
5. Rich visualisations: zone temperature timeseries, HVAC component states,
   energy rates, and spatial floor-plan heatmaps

## Architecture recap
```
WeatherController → TFSimulator.step_sim()
    ├─ finite_differences_timestep()   (conduction + convection)
    └─ per zone:
        VAV.output(zone_temp, supply_air_temp)
          ├─ if T < heating_sp → Boiler adds heat via reheat coil
          └─ if T > cooling_sp → AirHandler removes heat
            └─ building.apply_thermal_power_zone(zone, q_zone)
```

## Notebook contents
| Section | Content |
|---|---|
| 0 | Imports |
| 1 | Floor Plan Definition and Visualisation |
| 2 | Building Construction & HVAC Sizing |
| 3 | HVAC Configuration |
| 4 | Weather Controller |
| 5 | Occupancy Model (illustrative only) |
| 6–7 | Heating Scenario (3-day) + debug visualisation |
| 8–9 | Cooling Scenario (3-day) + debug visualisation |
| 10 | Scenario Comparison |
| 11 | Summary |

## Section 0 – Imports

Standard library, numeric/plotting stack, and SBSim simulator modules.
The cell also pins the protobuf backend and resolves the repo root on
`sys.path` so `smart_control.*` is importable regardless of how the
notebook is launched.

In [ ]:
import copy
import os
import sys

# smart_control/proto/ uses protoc-generated code that is incompatible with
# the protobuf 4.x C-extension backend.
os.environ.setdefault("PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION", "python")

import matplotlib
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Try __file__ (script mode), then VS Code's __vsc_ipynb_file__ (notebook
# mode in VS Code), then fall back to cwd as a last resort.
try:
  _nb_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
  try:
    _nb_dir = os.path.dirname(os.path.abspath(__vsc_ipynb_file__))  # noqa: F821
  except NameError:
    _nb_dir = os.getcwd()

REPO_ROOT = os.path.abspath(os.path.join(_nb_dir, "..", ".."))
if REPO_ROOT not in sys.path:
  sys.path.insert(0, REPO_ROOT)

# PORT (2026-08-17): this tree is Google's raw internal snapshot
# (internal_2026_38_15_11_5_2, published as PR #169). It is not runnable as
# checked out: 186 files import through the internal package roots
# `smart_buildings.*` and `google3.*`, no package has an __init__.py, and the
# generated *_pb2.py are absent (copybara exports strip them). The bootstrap
# maps those names onto this repo and generates nothing at run time. It is
# local-only and git-excluded -- see .git/info/exclude.
import sbsim_local_bootstrap  # noqa: F401  pylint: disable=unused-import

# pylint: disable=wrong-import-position
from smart_control.simulator import air_handler as air_handler_py
from smart_control.simulator import boiler as boiler_py
from smart_control.simulator import building as building_py
from smart_control.simulator import hot_water_system as hot_water_system_py
from smart_control.simulator import hvac_floorplan_based as floorplan_hvac_py
from smart_control.simulator import pump as pump_py
from smart_control.simulator import setpoint_schedule as setpoint_schedule_py
from smart_control.simulator import step_function_occupancy as occupancy_py
from smart_control.simulator import weather_controller as weather_controller_py
# pylint: enable=wrong-import-position

KELVIN_TO_CELSIUS = 273.15

print("All imports OK.")
print(f"NumPy {np.__version__}, Pandas {pd.__version__}, Matplotlib {matplotlib.__version__}")


## Section 1 – Floor Plan Definition and Visualisation

The floor plan is a 2-D NumPy integer array (23 rows × 12 cols) where each
cell represents one 20 cm × 20 cm control volume (CV):

| Code | Type | Location in grid |
|------|------|-----------------|
| `0` | Interior air – room space | rows 3–10 (`room_1`), rows 12–19 (`room_2`) |
| `1` | Interior wall / corridor | perimeter corridors + dividing wall at row 11 |
| `2` | Exterior boundary | outermost row and column on all four sides |

The two rooms are stacked vertically and share a single corridor wall at row 11.
Total: 96 air CVs · 114 interior wall CVs · 66 exterior CVs.

Grid taken from `FlexibleFloorplanSimulatorTest._create_dummy_floor_plan_small()`
in the test suite.

In [ ]:
# Floor plan taken verbatim from
# FlexibleFloorplanSimulatorTest._create_dummy_floor_plan_small().
# Codes: 0 = interior air, 1 = interior wall/corridor, 2 = exterior boundary.
FLOOR_PLAN = np.array([
    [2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2],  # row  0  exterior top
    [2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2],  # row  1  corridor (N)
    [2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2],  # row  2  corridor (N)
    [2, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 2],  # row  3  room_1  first air row
    [2, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 2],  # row  4  room_1
    [2, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 2],  # row  5  room_1
    [2, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 2],  # row  6  room_1  (air at cols 3-8)
    [2, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 2],  # row  7  room_1
    [2, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 2],  # row  8  room_1
    [2, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 2],  # row  9  room_1
    [2, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 2],  # row 10  room_1  last air row
    [2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2],  # row 11  dividing corridor wall
    [2, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 2],  # row 12  room_2  first air row
    [2, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 2],  # row 13  room_2
    [2, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 2],  # row 14  room_2
    [2, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 2],  # row 15  room_2  (air at cols 3-8)
    [2, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 2],  # row 16  room_2
    [2, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 2],  # row 17  room_2
    [2, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 2],  # row 18  room_2
    [2, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 2],  # row 19  room_2  last air row
    [2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2],  # row 20  corridor (S)
    [2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2],  # row 21  corridor (S)
    [2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2],  # row 22  exterior bottom
])

print(f"Floor plan shape: {FLOOR_PLAN.shape}  (rows × cols)")
print(f"Air cells  (code 0): {(FLOOR_PLAN == 0).sum()}")
print(f"Wall cells (code 1): {(FLOOR_PLAN == 1).sum()}")
print(f"Ext. cells (code 2): {(FLOOR_PLAN == 2).sum()}")

In [ ]:
fig_fp, ax_fp = plt.subplots(figsize=(7, 10))

# BoundaryNorm centres each integer code in its bin, preventing colour bleed
# at code boundaries when the array contains exactly 0, 1, and 2.
cmap_fp = mcolors.ListedColormap(["white", "#aaaaaa", "#444444"])
bounds_fp = [-0.5, 0.5, 1.5, 2.5]
norm_fp = mcolors.BoundaryNorm(bounds_fp, cmap_fp.N)

# origin="upper" aligns row 0 with the top of the image, matching NumPy
# array indexing convention.
ax_fp.imshow(
    FLOOR_PLAN, cmap=cmap_fp, norm=norm_fp, origin="upper", aspect="equal"
)

# Label midpoints: x=5.5 is the centre of the air block (cols 3-8);
# y=6.5 and y=15.5 are the vertical centres of room_1 (rows 3-10)
# and room_2 (rows 12-19) respectively.
ax_fp.text(
    5.5, 6.5, "room_1\n(zone 1)",
    ha="center", va="center", fontsize=12, color="steelblue", fontweight="bold",
)
ax_fp.text(
    5.5, 15.5, "room_2\n(zone 2)",
    ha="center", va="center", fontsize=12, color="darkorange", fontweight="bold",
)

legend_patches = [
    mpatches.Patch(color="white",   label="Air (code 0) – room space"),
    mpatches.Patch(color="#aaaaaa", label="Wall (code 1) – interior / corridor"),
    mpatches.Patch(color="#444444", label="Wall (code 2) – exterior envelope"),
]
ax_fp.legend(handles=legend_patches, loc="upper right", fontsize=9, framealpha=0.9)

ax_fp.set_title(
    "Two-Zone Office Building – Floor Plan\n"
    "(23 × 12 control-volume grid, cv_size = 20 cm)",
    fontsize=12,
)
ax_fp.set_xlabel("Column index (East–West)")
ax_fp.set_ylabel("Row index (North–South)")
plt.tight_layout()
plt.show()

## Section 2 – Building Construction & HVAC Sizing

### Material properties

Default properties from `building.py`, passed directly to `FloorPlanBasedBuilding`:

| Layer | Conductivity (W/m·K) | Heat cap (J/kg·K) | Density (kg/m³) | Class |
|-------|---------------------|-------------------|-----------------|-------|
| Air | 50.0 – artificially high to model convective mixing | 700 | 1 | `DefaultInsideAirMaterialProperties` |
| Int. wall | 2.0 | 1 000 | 1 800 | `DefaultInsideWallMaterialProperties` |
| Ext. wall | 0.05 | 1 000 | 3 000 | `DefaultExteriorWallMaterialProperties` |

### Building geometry

- **Air cells** (code 0): 96 CVs × (0.20 m)² → floor area = 3.84 m²
- **Floor height**: 3.0 m → building volume = 11.52 m³
- **Exterior cells** (code 2): 66 CVs → envelope area = 66 × 0.20 × 3.0 = 39.6 m²

### Rule-of-thumb HVAC sizing

Steady-state envelope heat loss (winter design, ΔT = 25 K):

$$Q_{loss} = U \times A_{env} \times \Delta T$$

Exterior wall U-value (k = 0.05 W/m·K, two CV layers → d = 0.40 m):

$$U = k/d = 0.05/0.40 = 0.125\ \text{W/m}^2\text{K}$$
$$Q_{loss} = 0.125 \times 39.6 \times 25 \approx 124\ \text{W}$$

The code applies an oversize factor of **15×** to handle thermal mass during warm-up:

$$Q_{design} = 15 \times 124 \approx 1{,}856\ \text{W} = 1.9\ \text{kW (heating per zone)}$$
$$Q_{design} = 15 \times 74 \approx 1{,}114\ \text{W} = 1.1\ \text{kW (cooling per zone)}$$

> **Note:** this sizing is informational only — it justifies the oversize factor but does not directly set device parameters. Actual boiler, AHU, and VAV values are configured in Section 3, where a sanity check reveals the VAV’s estimated zone supply capacity (~21.8 kW) greatly exceeds this design value.

In [ ]:
CV_SIZE_CM = 20.0        # Control-volume side length [cm].
FLOOR_HEIGHT_CM = 300.0  # Floor-to-ceiling height [cm].

# PORT (2026-08-17): PR #169 removed the Default*MaterialProperties helper
# subclasses; only the MaterialProperties dataclass remains. The values below
# are copied verbatim from the pre-port definitions (copybara_push
# building.py:33-53) so the envelope physics is unchanged.
inside_air_props = building_py.MaterialProperties(
    conductivity=50.0, heat_capacity=700.0, density=1.2
)
inside_wall_props = building_py.MaterialProperties(
    conductivity=2.0, heat_capacity=1000.0, density=1800.0
)
exterior_wall_props = building_py.MaterialProperties(
    conductivity=0.05, heat_capacity=1000.0, density=3000.0
)

print("=== Material Properties ===")
for label, props in [
    ("Air", inside_air_props),
    ("Interior wall", inside_wall_props),
    ("Exterior wall", exterior_wall_props),
]:
  print(
      f"  {label:15s}: k={props.conductivity:.2f} W/m·K, "
      f"c={props.heat_capacity:.0f} J/kg·K, "
      f"ρ={props.density:.0f} kg/m³"
  )


def create_building(initial_temp_k: float) -> building_py.FloorPlanBasedBuilding:
  """Creates a FloorPlanBasedBuilding initialised to a uniform temperature.

  Args:
    initial_temp_k: Starting temperature in Kelvin for all control volumes.

  Returns:
    A fully constructed FloorPlanBasedBuilding instance.
  """
  zone_map = copy.deepcopy(FLOOR_PLAN)
  return building_py.FloorPlanBasedBuilding(
      cv_size_cm=CV_SIZE_CM,
      floor_height_cm=FLOOR_HEIGHT_CM,
      initial_temp=initial_temp_k,
      inside_air_properties=inside_air_props,
      inside_wall_properties=inside_wall_props,
      building_exterior_properties=exterior_wall_props,
      floor_plan=FLOOR_PLAN,
      zone_map=zone_map,
      buffer_from_walls=0,
      # PORT (2026-08-17): PR #169 added a min_room_size gate to diffuser
      # placement (building.py:290, default 350 control volumes;
      # thermal_diffuser_utils.py:221 returns [] below it). This floor plan has
      # 48 CVs per room, so the default silently places ZERO diffusers --
      # Building.apply_thermal_power_zone only deposits heat where
      # diffusers > 0, so every watt of supply air would be discarded and the
      # baseline would look like a free-floating building. Lowered so the two
      # rooms are served, reproducing the pre-port placement exactly
      # (1 diffuser per room, weight 1.0).
      min_room_size=1,
  )


# Smoke-test: verify zone discovery and grid shape before running simulations.
_bld_test = create_building(293.0)
ROOM_ZONES = sorted(_bld_test.get_zone_average_temps())
print(f"\nActual room zones     : {ROOM_ZONES}")
print(f"Temperature grid shape: {_bld_test.temp.shape}")
print(f"Initial zone temps (K): {_bld_test.get_zone_average_temps()}")

# HVAC sizing calculations.
N_AIR_CELLS = int((FLOOR_PLAN == 0).sum())
N_ZONES = len(ROOM_ZONES)
CV_SIZE_M = CV_SIZE_CM / 100.0
FLOOR_HEIGHT_M = FLOOR_HEIGHT_CM / 100.0
FLOOR_AREA_M2 = N_AIR_CELLS * CV_SIZE_M ** 2
BUILDING_VOLUME_M3 = FLOOR_AREA_M2 * FLOOR_HEIGHT_M

# Each exterior cell contributes one wall face; corner cells contribute two,
# so add N_CORNER_CELLS once to correct the undercount.
N_EXTERIOR_CELLS = int((FLOOR_PLAN == 2).sum())
N_CORNER_CELLS = 4
ENVELOPE_AREA_M2 = (N_EXTERIOR_CELLS + N_CORNER_CELLS) * CV_SIZE_M * FLOOR_HEIGHT_M

# The exterior wall is one CV layer thick in this grid.
EXTERIOR_WALL_THICKNESS_M = CV_SIZE_M
U_WALL = exterior_wall_props.conductivity / EXTERIOR_WALL_THICKNESS_M

DESIGN_DT_HEATING = 25.0  # K: indoor 20 °C, outdoor -5 °C.
DESIGN_DT_COOLING = 15.0  # K: indoor 22 °C, outdoor 37 °C.

Q_LOSS_HEATING_W = U_WALL * ENVELOPE_AREA_M2 * DESIGN_DT_HEATING
Q_LOSS_COOLING_W = U_WALL * ENVELOPE_AREA_M2 * DESIGN_DT_COOLING

# Oversize factor accounts for thermal mass that must be overcome during
# warm-up / cool-down transients.
OVERSIZE_FACTOR = 15.0
Q_DESIGN_HEATING_W = Q_LOSS_HEATING_W * OVERSIZE_FACTOR
Q_DESIGN_COOLING_W = Q_LOSS_COOLING_W * OVERSIZE_FACTOR

print("\n=== HVAC Sizing ===")
print(f"  Air cells: {N_AIR_CELLS}, Zones: {N_ZONES}")
print(f"  Floor area: {FLOOR_AREA_M2:.2f} m²")
print(f"  Building volume: {BUILDING_VOLUME_M3:.2f} m³")
print(
    f"  Exterior CVs: {N_EXTERIOR_CELLS}, Envelope area: {ENVELOPE_AREA_M2:.1f} m²"
)
print(f"  Exterior wall thickness: {EXTERIOR_WALL_THICKNESS_M:.2f} m")
print(f"  U_wall: {U_WALL:.4f} W/m²·K")
print(
    f"  Steady-state heat loss (ΔT={DESIGN_DT_HEATING}K): "
    f"{Q_LOSS_HEATING_W:.0f} W"
)
print(
    f"  Design heating capacity ({OVERSIZE_FACTOR}×): "
    f"{Q_DESIGN_HEATING_W:.0f} W = {Q_DESIGN_HEATING_W/1000:.1f} kW"
)
print(
    f"  Design cooling capacity ({OVERSIZE_FACTOR}×): "
    f"{Q_DESIGN_COOLING_W:.0f} W = {Q_DESIGN_COOLING_W/1000:.1f} kW"
)

print(f"\n=== Diffuser Check ===")
print(f"  Diffuser sum: {_bld_test.diffusers.sum():.3f}")
print(f"  Non-zero diffuser cells: {(_bld_test.diffusers > 0).sum()}")
for z in ROOM_ZONES:
  zone_coords = _bld_test._room_dict[z]  # pylint: disable=protected-access
  diff_sum = sum(
      _bld_test.diffusers[c] for c in zone_coords if _bld_test.diffusers[c] > 0
  )
  n_diff = sum(1 for c in zone_coords if _bld_test.diffusers[c] > 0)
  print(f"  {z}: {n_diff} diffusers, weight sum = {diff_sum:.4f}")

## Section 3 – HVAC Configuration (sized to building)

### HVAC hierarchy
```
FloorPlanBasedHvac
├── Boiler        – hot water loop for VAV reheat
├── AirHandler    – mixed / conditioned supply air
└── VAV × N zones – variable-air-volume with thermostat
      └── SetpointSchedule – comfort (9am–6pm) vs eco
```

### Sizing approach

From Section 2 we computed the design heating load.  The VAV delivers
heat to each zone via:

$$q_{zone} = \dot{m}_{air} \cdot c_{air} \cdot (T_{supply} - T_{zone})$$

The simulator treats `max_air_flow_rate` as volumetric (m³/s) and
multiplies by `AIR_HEAT_CAPACITY = 1006 J/kg·K` directly (i.e. assumes
ρ ≈ 1 kg/m³ implicitly).  We size the VAV air and water flows so that
the per-zone capacity ≈ `Q_DESIGN_HEATING_W / N_ZONES`.

| Device | Parameter | Value | Rationale |
|--------|-----------|-------|-----------|
| Boiler | `reheat_water_setpoint` | 350 K (77 °C) | Standard HW |
| AHU | `heating_air_temp_setpoint` | 292 K | Min supply air |
| AHU | `cooling_air_temp_setpoint` | 300 K | Max supply air |
| AHU | `recirculation` | 0.3 | 30 % return air |
| AHU | `max_air_flow_rate` | 2.0 m³/s | Building total |
| VAV | `max_air_flow_rate` | 0.50 m³/s | Per zone |
| VAV | `reheat_max_water_flow_rate` | 0.10 kg/s | Per zone |

In [ ]:
# ---------------------------------------------------------------------------
# PORT (2026-08-17) to the PR #169 / internal_2026_38_15_11_5_2 architecture.
#
# What changed on the water side. The Boiler no longer owns the pump and is no
# longer handed to the Hvac directly. It is now a *heat source* inside a
# HotWaterSystem, which also owns a WaterPump and computes the hydraulics:
#
#     Boiler(supply_water_temperature_setpoint=...)      <- was reheat_water_setpoint
#     WaterPump(water_pump_differential_head=..., water_pump_efficiency=...)
#     HotWaterSystem(heat_source=boiler, pump=pump, header_resistance=...)
#     FloorPlanBasedHvac(hot_water_system=...)           <- was boiler=
#
# HotWaterSystem also accepts an AirSourceHeatPump as its heat source
# (HeatSourceType.ASHP, new in this tree), which is why the reward proto gained
# HeatPumpRewardInfo.
#
# What changed on the air side. The AirHandler's two deck setpoints collapsed
# into ONE supply_air_temperature_setpoint, and fan_differential_pressure was
# renamed fan_static_pressure. Google's own docstring: "in the real building,
# only the supply_air_temperature_setpoint exists". Consequence for this
# notebook: a single AHU can no longer serve heating and cooling at once, so
# create_hvac() now takes the supply-air setpoint as an argument and each
# scenario builds its own HVAC. The 2026-07-28 deck-ordering guard
# (AHU_HEAT_SP_K < AHU_COOL_SP_K < COOLING_SETPOINT_K) is therefore obsolete
# and has been removed -- the first inequality no longer exists, and the second
# is re-expressed below as a per-scenario check.
#
# What changed in VAV water sizing. vav_reheat_max_water_flow_rate [kg/s] is
# now vav_reheat_max_water_flow_factor -- a valve flow COEFFICIENT (Kv), not a
# flow. The system solves hydraulically (hot_water_system.py:301):
#
#     total_flow = Kv_sum * sqrt(dp / (1 + Kv_sum^2 * header_resistance))
#     dp [bar]   = pump_head [m] * 0.0981          (pump.py:101)
#
# and each VAV draws total_flow * (its Kv / Kv_sum). With header_resistance =
# 0 this reduces to per-VAV flow = Kv * sqrt(dp), independent of zone count.
# The factor below is therefore DERIVED from the old per-zone flow rather than
# carried over as a number, so the ported baseline is hydraulically equivalent
# to the pre-port one.
# ---------------------------------------------------------------------------

# Thermostat comfort band and eco (setback) windows.
HEATING_SETPOINT_K = 292.0  # 18.85 °C
COOLING_SETPOINT_K = 297.0  # 23.85 °C
ECO_LOW_K = 288.0           # 14.85 °C
ECO_HIGH_K = 300.0          # 26.85 °C

# Hot-water system parameters.
BOILER_SUPPLY_WATER_SP_K = 350.0  # 77 °C: standard hot-water loop temperature.
BOILER_PUMP_HEAD = 3.0            # [m]
BOILER_PUMP_EFFICIENCY = 0.6
HWS_HEADER_RESISTANCE = 0.0       # Default; keeps flow linear in Kv.

# Air handler parameters. ONE supply-air setpoint in this tree.
AHU_RECIRCULATION = 0.3    # Fraction of RETURN air blended with outdoor air,
                           # i.e. 30 % return / 70 % outdoor. Note that SBSim's
                           # own gin configs comment this as "percentage of
                           # fresh air", which contradicts air_handler.py:199.
AHU_SUPPLY_AIR_SP_HEATING_K = 292.0  # 18.85 °C. Was AHU_HEAT_SP_K (hot deck).
AHU_SUPPLY_AIR_SP_COOLING_K = 294.0  # 20.85 °C. Was AHU_COOL_SP_K (cold deck);
                                     # must stay below COOLING_SETPOINT_K or
                                     # mechanical cooling can never reach the
                                     # zone setpoint (the 2026-07-28 finding).
AHU_FAN_PRESSURE = 500.0   # [Pa] Typical duct static pressure.
AHU_FAN_EFFICIENCY = 0.8

# ---------------------------------------------------------------------------
# FIX (2026-07-28), retained: VAV flows are sized from the building's own
# computed design capacity instead of being hard-coded.
#
# The previous values (0.50 kg/s air, 0.10 kg/s water per zone) were sized for
# a full-size building, but this floor plan is only 3.84 m2 / 11.5 m3. They
# produced ~8.6 kW of average boiler gas against a computed steady-state
# envelope loss of 262 W -- roughly 33x the actual load, and 2.2x the
# notebook's own design capacity figure.
#
# Note on units: SBSim multiplies these flows directly by AIR_HEAT_CAPACITY
# [J/kg/K] and WATER_HEAT_CAPACITY [J/kg/K], so both are MASS flows in kg/s
# despite the "m3/s" / "m3/h" labels used in Google's code and gin configs.
# ---------------------------------------------------------------------------
DESIGN_HEATING_W = 3938.0            # From the Section 2 sizing calculation.
N_ZONES = len(ROOM_ZONES)
ZONE_DESIGN_W = DESIGN_HEATING_W / N_ZONES

# Air side: size to carry the design load at a 22 K supply-to-zone rise.
DESIGN_AIR_RISE_K = 22.0
VAV_MAX_AIR_FLOW = 0.50   # [kg/s] per zone. Original value, restored.
_SIZED_AIR_FLOW = round(ZONE_DESIGN_W / (1006.0 * DESIGN_AIR_RISE_K), 3)

# Water side: the pre-port notebook used 0.10 kg/s per zone. Convert that to
# the Kv the new hydraulic solver needs, so behaviour is preserved exactly.
VAV_TARGET_WATER_FLOW = 0.10  # [kg/s] per zone. Original value, restored.
BAR_PER_METRE_HEAD = 0.0981   # pump.py:101
_PUMP_DP_BAR = BOILER_PUMP_HEAD * BAR_PER_METRE_HEAD
VAV_REHEAT_MAX_WATER_FLOW_FACTOR = VAV_TARGET_WATER_FLOW / np.sqrt(_PUMP_DP_BAR)

# AHU must be able to carry every VAV at full damper, with headroom.
AHU_MAX_AIR_FLOW = 2.0    # [kg/s] total. Original value, restored.
print("  [sizing note] envelope-derived air flow would be "
      f"{_SIZED_AIR_FLOW} kg/s per zone; in use: {VAV_MAX_AIR_FLOW}.")
print("  That gap IS the oversizing finding -- do not close it until the "
      "plateau is diagnosed.")
print(f"  [port] pump dp = {_PUMP_DP_BAR:.4f} bar -> reheat Kv = "
      f"{VAV_REHEAT_MAX_WATER_FLOW_FACTOR:.4f} reproduces "
      f"{VAV_TARGET_WATER_FLOW} kg/s per zone.")


def zone_vav(hvac, zone_id):
  """Returns the single Vav serving zone_id.

  PORT (2026-08-17): PR #169 rekeyed Hvac.vavs from ZONE id ('room_1') to VAV
  DEVICE id ('VAV_1'), and a zone may now map to a LIST of VAVs
  (hvac_floorplan_based.py:123 defaults to {room_N: [VAV_N]}). This floor plan
  has exactly one VAV per zone, so the helper asserts that and unwraps it.
  Indexing hvac.vavs by zone id -- as the pre-port notebook did -- now raises
  KeyError.

  Args:
    hvac: A FloorPlanBasedHvac instance.
    zone_id: Zone name as used by the building, e.g. 'room_1'.

  Returns:
    The Vav serving that zone.
  """
  vav_ids = hvac._zone_to_vav_ids[zone_id]  # pylint: disable=protected-access
  assert len(vav_ids) == 1, f'{zone_id} maps to {len(vav_ids)} VAVs, expected 1'
  return hvac.vavs[vav_ids[0]]


def create_hvac(
    zone_identifiers, supply_air_setpoint_k: float
) -> floorplan_hvac_py.FloorPlanBasedHvac:
  """Creates a FloorPlanBasedHvac wired to the given zone identifiers.

  Args:
    zone_identifiers: Sequence of zone name strings that must match the
      building's zone map keys.
    supply_air_setpoint_k: The AHU's single supply-air temperature setpoint in
      K. In this architecture one AHU cannot serve heating and cooling at the
      same time, so each scenario supplies its own value.

  Returns:
    A fully constructed FloorPlanBasedHvac with a hot-water system (boiler +
    pump), an air handler, and one VAV per zone.
  """
  new_boiler = boiler_py.Boiler(
      supply_water_temperature_setpoint=BOILER_SUPPLY_WATER_SP_K,
      device_id="boiler_id",
      # PORT (2026-08-17): supersedes the 2026-07-28 post-construction fix.
      # The return-water sensor is now an explicit constructor argument that
      # Boiler.reset() restores to, so it survives Hvac.__init__'s reset()
      # instead of being silently zeroed.
      init_return_water_temperature_sensor=BOILER_SUPPLY_WATER_SP_K,
  )

  new_pump = pump_py.WaterPump(
      water_pump_differential_head=BOILER_PUMP_HEAD,
      water_pump_efficiency=BOILER_PUMP_EFFICIENCY,
      device_id="pump_id",
  )

  new_hws = hot_water_system_py.HotWaterSystem(
      heat_source=new_boiler,
      pump=new_pump,
      device_id="hws_id",
      header_resistance=HWS_HEADER_RESISTANCE,
  )

  new_hvac = floorplan_hvac_py.FloorPlanBasedHvac(
      zone_identifier=zone_identifiers,
      air_handler=air_handler_py.AirHandler(
          recirculation=AHU_RECIRCULATION,
          supply_air_temperature_setpoint=supply_air_setpoint_k,
          fan_static_pressure=AHU_FAN_PRESSURE,
          fan_efficiency=AHU_FAN_EFFICIENCY,
          max_air_flow_rate=AHU_MAX_AIR_FLOW,
      ),
      hot_water_system=new_hws,
      schedule=setpoint_schedule_py.SetpointSchedule(
          morning_start_hour=9,
          evening_start_hour=18,
          comfort_temp_window=(HEATING_SETPOINT_K, COOLING_SETPOINT_K),
          eco_temp_window=(ECO_LOW_K, ECO_HIGH_K),
          holidays={7, 223, 245},
      ),
      vav_max_air_flow_rate=VAV_MAX_AIR_FLOW,
      vav_reheat_max_water_flow_factor=VAV_REHEAT_MAX_WATER_FLOW_FACTOR,
  )

  return new_hvac


# ---------------------------------------------------------------------------
# Sanity check.
#
# FIX (2026-07-28), retained and re-verified against this tree: the zone supply
# temperature is NOT a capacity-weighted mean of the air and water streams.
# vav.py:242 computes
#     T_out = [T_air*(C_air - C_water) + T_water*C_water] / C_air
#           = T_air + (C_water / C_air) * (T_water - T_air)
# which is unbounded above: whenever C_water > C_air the supply air leaves
# HOTTER than the hot water feeding it, violating the second law. The formula
# is algebraically identical to the pre-port one, so this finding SURVIVES the
# #169 rework and is still worth raising with Google.
# ---------------------------------------------------------------------------
print("\n=== HVAC Sanity Check ===")
c_air_flow = 1006.0 * VAV_MAX_AIR_FLOW
c_water_flow = 4180.0 * VAV_TARGET_WATER_FLOW
capacity_ratio = c_water_flow / c_air_flow
ahu_supply_approx = AHU_SUPPLY_AIR_SP_HEATING_K

# What SBSim actually computes (vav.py:242).
zone_supply_sim = ahu_supply_approx + capacity_ratio * (
    BOILER_SUPPLY_WATER_SP_K - ahu_supply_approx
)
# Physically correct bound: air can at most reach the water temperature.
zone_supply_bound = min(zone_supply_sim, BOILER_SUPPLY_WATER_SP_K)
q_heat_sim = c_air_flow * (zone_supply_sim - 275.0)

print(f"  Zones: {N_ZONES}, design heating: {DESIGN_HEATING_W/1000:.2f} kW"
      f" ({ZONE_DESIGN_W/1000:.2f} kW/zone)")
print(f"  VAV_MAX_AIR_FLOW   = {VAV_MAX_AIR_FLOW} kg/s per zone")
print(f"  VAV water flow     = {VAV_TARGET_WATER_FLOW} kg/s per zone"
      f" (Kv = {VAV_REHEAT_MAX_WATER_FLOW_FACTOR:.4f})")
print(f"  AHU_MAX_AIR_FLOW   = {AHU_MAX_AIR_FLOW} kg/s total")
print(f"  C_air*flow = {c_air_flow:.0f} W/K   "
      f"C_water*flow = {c_water_flow:.0f} W/K")
print(f"  Capacity ratio C_water/C_air = {capacity_ratio:.3f}")
if capacity_ratio >= 1.0:
  print("  *** WARNING: ratio >= 1. Vav.compute_zone_supply_temp will return")
  print("      air HOTTER than the reheat water (second-law violation).")
  print("      Google's shipped sb1 config has ratio 3.56 and does exactly")
  print("      this. Reduce the reheat Kv or raise VAV_MAX_AIR_FLOW.")
else:
  print("  OK: ratio < 1, so the VAV reheat formula stays physical here.")
print(f"  Zone supply temp, SBSim formula : "
      f"{zone_supply_sim - KELVIN_TO_CELSIUS:.1f} °C")
print(f"  Zone supply temp, physical bound: "
      f"{zone_supply_bound - KELVIN_TO_CELSIUS:.1f} °C"
      f"  (boiler water = {BOILER_SUPPLY_WATER_SP_K - KELVIN_TO_CELSIUS:.1f} °C)")
print(f"  Estimated q_zone from 275K: {q_heat_sim/1000:.1f} kW")

# PORT (2026-08-17): replaces the two-deck guard. With a single supply-air
# setpoint the only remaining requirement is that the COOLING scenario's
# setpoint sits below the zone cooling setpoint -- otherwise mechanical
# cooling can never bring a zone to setpoint, which is the defect that
# produced 0 % cooling comfort before 2026-07-28.
if AHU_SUPPLY_AIR_SP_COOLING_K >= COOLING_SETPOINT_K:
  print("  *** WARNING: cooling supply-air setpoint >= COOLING_SETPOINT_K.")
  print("      Mechanical cooling can never reach the zone setpoint.")
else:
  print(f"  OK: cooling supply air"
        f" ({AHU_SUPPLY_AIR_SP_COOLING_K - KELVIN_TO_CELSIUS:.1f} °C) is below"
        f" the zone cooling setpoint"
        f" ({COOLING_SETPOINT_K - KELVIN_TO_CELSIUS:.1f} °C).")

# Smoke-test the heating configuration and confirm the hydraulics resolve.
_hvac_test = create_hvac(ROOM_ZONES, AHU_SUPPLY_AIR_SP_HEATING_K)
_hws_test = _hvac_test.hot_water_system
print("  HWS supply water SP: "
      f"{_hws_test.supply_water_temperature_setpoint - KELVIN_TO_CELSIUS:.1f} °C")
print("  HWS init RW sensor:  "
      f"{_hws_test.return_water_temperature_sensor - KELVIN_TO_CELSIUS:.1f} °C"
      "  (should equal the supply SP, not -273.1)")
# The hot-water system accumulates flow demand PER STEP: the simulator calls
# reset_demand() and then add_demand(vav.reheat_flow_factor) for every VAV
# (simulator.py:443). Outside a step Kv_sum is therefore 0 and total_flow_rate
# reads 0 -- that is correct, not a mis-sizing. Reproduce one step's
# accumulation with the valves fully open to confirm the derived Kv really
# delivers the intended per-zone flow.
_hws_test.reset_demand()
for _v in _hvac_test.vavs.values():
  _v.reheat_valve_setting = 1.0
  _hws_test.add_demand(_v.reheat_flow_factor)
print(f"  HWS pump dp: {_hws_test.differential_pressure_setpoint:.4f} bar, "
      f"Kv_sum = {_hws_test.flow_factor_sum:.4f} (valves fully open)")
print(f"  HWS total flow: {_hws_test.total_flow_rate:.4f} kg/s "
      f"-> {_hws_test.total_flow_rate / N_ZONES:.4f} kg/s per zone "
      f"(target {VAV_TARGET_WATER_FLOW})")


## HVAC Device Roles and Temperatures

**Temperatures (from code)**

| | Heating Scenario | Cooling Scenario |
|---|---|---|
| Building start | 1.85°C | 36.85°C |
| Outdoor air | -5°C to 4°C | 22°C to 35°C |
| Comfort target | 18.85°C – 23.85°C | 18.85°C – 23.85°C |

---

**Heating Scenario**

| Device | Role | Temperature |
|---|---|---|
| Boiler | Heats water, pumps to VAV reheat coils | Supply water: 77°C |
| AHU | Mixes outdoor + return air, heats to setpoint, sends to ducts | Supply air: 18.85°C |
| VAV (each zone) | Opens damper + reheat valve, reheat coil mixes AHU air with boiler water | Zone supply air: ~45°C |
| Zone | Receives 45°C air, temperature rises toward 18.85°C | Start: 1.85°C → Target: 18.85°C |

---

**Cooling Scenario**

| Device | Role | Temperature |
|---|---|---|
| Boiler | Idle | — |
| AHU | Mixes outdoor + return air, cools to setpoint, sends to ducts | Supply air: 26.85°C |
| VAV (each zone) | Opens damper only, reheat valve closed | Zone supply air: 26.85°C |
| Zone | Receives 26.85°C air, temperature slowly drops | Start: 36.85°C → Target: 23.85°C |

---

> **Note:** The cooling scenario has a fundamental problem — the AHU supplies air at 26.85°C but the target upper comfort bound is 23.85°C. The AHU cannot cool the zone below its own supply temperature, so reaching the comfort band is physically impossible with this setup. Any cooling that occurs is passive heat loss through the envelope, not active cooling.

## Section 4 – Weather Controller

`WeatherController` generates a sinusoidal outdoor temperature profile that
oscillates between a daily low and high over a 24-hour period. A convection
coefficient of 12.0 W/m²·K governs heat exchange between exterior boundary
cells and the outdoor air at each simulation step.

| Scenario | Dates | Daily low | Daily high |
|---|---|---|---|
| Winter | Jan 10–12, 2024 | 268 K (−5 °C) | 277 K (4 °C) |
| Summer | Jul 15–17, 2024 | 295 K (22 °C) | 308 K (35 °C) |

The plot below overlays the outdoor temperature profile against the comfort
setpoints to show how far each scenario sits from the comfort band.

In [ ]:
wc_winter = weather_controller_py.WeatherController(
    default_low_temp=268.0,   # -5 °C
    default_high_temp=277.0,  # 4 °C
    convection_coefficient=12.0,
)
wc_summer = weather_controller_py.WeatherController(
    default_low_temp=295.0,   # 22 °C
    default_high_temp=308.0,  # 35 °C
    convection_coefficient=12.0,
)

ts_range_w = pd.date_range("2024-01-10", periods=3 * 24 * 12, freq="5min")
winter_temps = [
    wc_winter.get_current_temp(t) - KELVIN_TO_CELSIUS for t in ts_range_w
]
ts_range_s = pd.date_range("2024-07-15", periods=3 * 24 * 12, freq="5min")
summer_temps = [
    wc_summer.get_current_temp(t) - KELVIN_TO_CELSIUS for t in ts_range_s
]

h_sp_c = HEATING_SETPOINT_K - KELVIN_TO_CELSIUS
c_sp_c = COOLING_SETPOINT_K - KELVIN_TO_CELSIUS

fig_wx, axes_wx = plt.subplots(2, 1, figsize=(14, 6), sharex=False)
for ax_wx, ts_rng, temps, colour, ttl in [
    (axes_wx[0], ts_range_w, winter_temps, "steelblue", "Winter (Jan 10–12, 2024) — Heating Scenario"),
    (axes_wx[1], ts_range_s, summer_temps, "darkorange", "Summer (Jul 15–17, 2024) — Cooling Scenario"),
]:
  ax_wx.plot(ts_rng, temps, color=colour, lw=1.5, label="Outdoor temp")
  ax_wx.axhline(
      h_sp_c, color="red", ls="--", lw=1, label=f"Heating SP ({h_sp_c:.2f}°C)"
  )
  ax_wx.axhline(
      c_sp_c, color="orange", ls="--", lw=1, label=f"Cooling SP ({c_sp_c:.2f}°C)",
  )
  ax_wx.set_title(ttl, fontsize=13)
  ax_wx.set_ylabel("Temperature [°C]", fontsize=11)
  ax_wx.legend(fontsize=10)
  ax_wx.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

## Section 5 – Occupancy Model

An illustrative step-function occupancy schedule is constructed here but is
**not coupled into the thermal solver**. It is included to show how occupancy
data would be wired in a full RL training loop — for example as an observation
feature or as a component of the reward signal.

The schedule steps between a peak occupancy of **10 persons/zone** during
business hours (09:00–17:00) and a background level of **0.1 persons/zone**
outside those hours.

`TFSimulator` is also imported at the end of this cell to defer TensorFlow
initialisation (and its GPU-discovery warnings) until after the occupancy
plot, keeping the import block output clean.

In [ ]:
occupancy = occupancy_py.StepFunctionOccupancy(
    work_start_time=pd.Timedelta(9, unit="h"),
    work_end_time=pd.Timedelta(17, unit="h"),
    work_occupancy=10.0,
    nonwork_occupancy=0.1,
)

occ_times = pd.date_range("2024-01-10", periods=2 * 24 * 12, freq="5min")
occ_vals = [
    occupancy.average_zone_occupancy("room_1", occ_times[i], occ_times[i + 1])
    for i in range(len(occ_times) - 1)
]

fig_occ, ax_occ = plt.subplots(figsize=(12, 3))
ax_occ.plot(
    occ_times[:-1],
    occ_vals,
    color="teal",
    lw=2,
    drawstyle="steps-post",
    label="Occupancy",
)
ax_occ.set_ylabel("Avg Occupants/Zone")
ax_occ.set_ylim(-0.5, 12)
ax_occ.set_title("Occupancy – 2-Day Sample", fontsize=11)
ax_occ.legend(fontsize=9)
ax_occ.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

# TFSimulator is imported here rather than at the top because it pulls in
# TensorFlow, which emits GPU-discovery warnings that are easier to read
# after the occupancy plot than interleaved with the import block.
# pylint: disable=wrong-import-position
from smart_control.simulator import tf_simulator as tf_simulator_py
# pylint: enable=wrong-import-position

## Section 6 – TFSimulator: Heating Scenario (3-day)

Runs a 3-day winter simulation (Jan 10–12) starting from a uniform building
temperature of **275 K (1.85 °C)**. The boiler and VAV reheat coils are
expected to overcome the thermal mass and bring both zones into the comfort
band (18.85 °C – 23.85 °C).

At each 5-minute step the simulation loop records:
- Zone average temperatures and active thermostat mode
- VAV damper and reheat valve positions per zone
- VAV zone supply temperature (after reheat coil mixing)
- Heat flux applied to each zone [kW]
- AHU supply air temperature and total air flow rate
- Boiler supply and return water temperatures
- Boiler gas consumption and fan electrical power [kW]
- Outdoor air temperature

These signals are plotted in Section 7.

In [ ]:
INITIAL_TEMP_HEATING = 275.0  # K: 1.85 °C cold start.
START_TS_WINTER = pd.Timestamp("2024-01-10 00:00:00")
N_DAYS = 3
TIME_STEP_SEC = 300.0
N_STEPS = int(N_DAYS * 24 * 3600 / TIME_STEP_SEC)

bld_heat = create_building(INITIAL_TEMP_HEATING)
# PORT (2026-08-17): the AHU now has a single supply-air setpoint, so each
# scenario builds its HVAC with the setpoint appropriate to that mode.
hvac_heat = create_hvac(
    sorted(bld_heat.get_zone_average_temps()), AHU_SUPPLY_AIR_SP_HEATING_K
)
ZONE_IDS = sorted(bld_heat.get_zone_average_temps())
REF_ZONE = ZONE_IDS[0]

sim_heat = tf_simulator_py.TFSimulator(
    building=bld_heat,
    hvac=hvac_heat,
    weather_controller=wc_winter,
    time_step_sec=TIME_STEP_SEC,
    convergence_threshold=0.1,
    iteration_limit=100,
    iteration_warning=50,
    start_timestamp=START_TS_WINTER,
)

print(
    "TFSimulator HEATING init. Mean zone T = "
    f"{bld_heat.temp.mean() - KELVIN_TO_CELSIUS:.2f} °C"
)

# Per-step data storage.
ts_heat = []
zone_temp_h = {z: [] for z in ZONE_IDS}
h_sp_h, c_sp_h, mode_h = [], [], []
damper_h = {z: [] for z in ZONE_IDS}
reheat_v_h = {z: [] for z in ZONE_IDS}
zone_supply_temp_h = {z: [] for z in ZONE_IDS}
q_zone_h = {z: [] for z in ZONE_IDS}
ahu_supply_h = []
ahu_flow_h = []
boiler_sw_h = []
boiler_rw_h = []
boiler_gas_h = []
fan_elec_h = []
outdoor_h = []

for step in range(N_STEPS):
  ts = sim_heat.current_timestamp
  # FIX (2026-07-28): set damper/valve for THIS step before logging.
  sim_heat.setup_step_sim()
  outdoor_k = wc_winter.get_current_temp(ts)
  # Use the mean grid temperature as a proxy for the AHU return-air sensor.
  rec_temp = bld_heat.temp.mean()
  ahu_supply_k = hvac_heat.air_handler.get_supply_air_temp(rec_temp, outdoor_k)

  ts_heat.append(ts)
  sched = zone_vav(hvac_heat, REF_ZONE).thermostat.get_setpoint_schedule()
  h_sp, c_sp = sched.get_temperature_window(ts)
  h_sp_h.append(h_sp)
  c_sp_h.append(c_sp)
  # pylint: disable=protected-access
  mode_h.append(zone_vav(hvac_heat, REF_ZONE).thermostat._current_mode.value)
  # pylint: enable=protected-access

  zone_avgs = bld_heat.get_zone_average_temps()
  for zid in ZONE_IDS:
    zone_temp_h[zid].append(zone_avgs[zid] - KELVIN_TO_CELSIUS)
    damper_h[zid].append(zone_vav(hvac_heat, zid).damper_setting)
    reheat_v_h[zid].append(zone_vav(hvac_heat, zid).reheat_valve_setting)
    zst_k = zone_vav(hvac_heat, zid).compute_zone_supply_temp(
        ahu_supply_k, hvac_heat.hot_water_system.supply_water_temperature_setpoint
    )
    zone_supply_temp_h[zid].append(zst_k - KELVIN_TO_CELSIUS)
    q_w = zone_vav(hvac_heat, zid).compute_energy_applied_to_zone(
        zone_avgs[zid], ahu_supply_k, hvac_heat.hot_water_system.supply_water_temperature_setpoint
    )
    q_zone_h[zid].append(q_w / 1000)

  ahu_supply_h.append(ahu_supply_k - KELVIN_TO_CELSIUS)
  ahu_flow_h.append(hvac_heat.air_handler.air_flow_rate)

  rw_k = hvac_heat.hot_water_system.return_water_temperature_sensor
  boiler_sw_h.append(
      hvac_heat.hot_water_system.supply_water_temperature_sensor - KELVIN_TO_CELSIUS
  )
  boiler_rw_h.append(rw_k - KELVIN_TO_CELSIUS)
  boiler_gas_h.append(
      hvac_heat.hot_water_system.compute_thermal_energy_rate(rw_k, outdoor_k) / 1000
  )
  fan_elec_h.append(
      (
          hvac_heat.air_handler.compute_intake_fan_energy_rate()
          + hvac_heat.air_handler.compute_exhaust_fan_energy_rate()
      )
      / 1000
  )
  outdoor_h.append(outdoor_k - KELVIN_TO_CELSIUS)

  sim_heat.execute_step_sim()

  if (step + 1) % 288 == 0:
    avg_t = sum(bld_heat.get_zone_average_temps().values()) / len(ZONE_IDS)
    print(
        f'  Day {(step+1)//288} | {ts.strftime("%m-%d %H:%M")}'
        f" | mean T={avg_t - KELVIN_TO_CELSIUS:.2f} °C"
        f" | outdoor={outdoor_k - KELVIN_TO_CELSIUS:.2f} °C"
        f" | mode={mode_h[-1]}"
    )

print("Heating simulation complete.")

## Section 7 – Heating Scenario: 12-Row Debug Visualisation

A stacked 12-panel figure covering all key HVAC signals for the winter run.
Both zones (`room_1` = blue, `room_2` = orange) are overlaid on each panel.

| Row | Signal | Notes |
|-----|--------|-------|
| 0 | Zone air temperature vs. setpoints | Red dashed = heating SP, orange dashed = cooling SP |
| 1 | Thermostat mode | 0 = OFF, 1 = HEAT, 2 = COOL, 3 = PASSIVE_COOL |
| 2 | VAV damper position [0–1] | 1 = fully open |
| 3 | VAV reheat valve position [0–1] | 1 = full hot-water flow |
| 4 | VAV zone supply temperature | Temperature of air entering the zone after the reheat coil |
| 5 | Heat injected to zone [kW] | Positive = heating, negative = cooling |
| 6 | AHU supply air temperature | Mixed outdoor + return air before VAV distribution |
| 7 | AHU total air flow rate [m³/s] | Aggregate flow for the whole building |
| 8 | Boiler supply water temperature | Should track 77 °C setpoint |
| 9 | Boiler return water temperature | Rises as zones absorb heat |
| 10 | HVAC energy rates [kW] | Boiler gas (red) + fan electricity (blue) |
| 11 | Outdoor air temperature | Driven by `wc_winter` sinusoidal profile |

In [ ]:
ZONE_COLORS = {"room_1": "steelblue", "room_2": "darkorange"}
TS_H = ts_heat

N_ROWS = 12
fig_h, axes_h = plt.subplots(N_ROWS, 1, figsize=(18, N_ROWS * 3), sharex=True)
fig_h.suptitle(
    "Heating Scenario (Winter) – 3-Day Debug\n"
    f"(VAV air={VAV_MAX_AIR_FLOW} m³/s, water={VAV_TARGET_WATER_FLOW} kg/s, "
    f"boiler={BOILER_SUPPLY_WATER_SP_K - KELVIN_TO_CELSIUS:.0f} °C)",
    fontsize=16,
)

# Row 0: Zone temperature vs. setpoints
ax = axes_h[0]
for zid in ZONE_IDS:
  ax.plot(TS_H, zone_temp_h[zid], color=ZONE_COLORS[zid], lw=1.5, label=zid)
ax.plot(
    TS_H,
    [t - KELVIN_TO_CELSIUS for t in h_sp_h],
    color="red",
    lw=1.2,
    ls="--",
    drawstyle="steps-post",
    label="Heating SP",
)
ax.plot(
    TS_H,
    [t - KELVIN_TO_CELSIUS for t in c_sp_h],
    color="orange",
    lw=1.2,
    ls="--",
    drawstyle="steps-post",
    label="Cooling SP",
)
ax.set_ylabel("Temp [°C]", fontsize=11)
ax.set_title("Zone Air Temperature vs. Setpoints", fontsize=13)
ax.legend(fontsize=15, ncol=4)
ax.grid(True, alpha=0.3)
ax.tick_params(axis="both", labelsize=10)

# Row 1: Thermostat mode
ax = axes_h[1]
ax.plot(
    TS_H,
    mode_h,
    color="purple",
    lw=1.5,
    drawstyle="steps-post",
    label="Mode (0=OFF,1=HEAT,2=COOL,3=PASSIVE_COOL)",
)
ax.set_ylabel("Mode", fontsize=11)
ax.set_yticks([0, 1, 2, 3])
ax.set_yticklabels(["OFF", "HEAT", "COOL", "P.COOL"])
ax.set_title("Thermostat Mode", fontsize=13)
ax.legend(fontsize=15)
ax.grid(True, alpha=0.3)
ax.tick_params(axis="both", labelsize=10)

# Row 2: VAV damper
ax = axes_h[2]
for zid in ZONE_IDS:
  ax.plot(
      TS_H,
      damper_h[zid],
      color=ZONE_COLORS[zid],
      lw=1.2,
      drawstyle="steps-post",
      label=f"{zid} damper",
  )
ax.set_ylabel("Damper [0-1]", fontsize=11)
ax.set_ylim(-0.05, 1.05)
ax.set_title("VAV Damper Position", fontsize=13)
ax.legend(fontsize=15, ncol=2)
ax.grid(True, alpha=0.3)
ax.tick_params(axis="both", labelsize=10)

# Row 3: VAV reheat valve
ax = axes_h[3]
for zid in ZONE_IDS:
  ax.plot(
      TS_H,
      reheat_v_h[zid],
      color=ZONE_COLORS[zid],
      lw=1.2,
      drawstyle="steps-post",
      label=f"{zid} reheat valve",
  )
ax.set_ylabel("Reheat Valve [0-1]", fontsize=11)
ax.set_ylim(-0.05, 1.05)
ax.set_title("VAV Reheat Valve Position", fontsize=13)
ax.legend(fontsize=15, ncol=2)
ax.grid(True, alpha=0.3)
ax.tick_params(axis="both", labelsize=10)

# Row 4: Zone supply temperature
ax = axes_h[4]
for zid in ZONE_IDS:
  ax.plot(
      TS_H,
      zone_supply_temp_h[zid],
      color=ZONE_COLORS[zid],
      lw=1.5,
      label=f"{zid} zone supply",
  )
ax.axhline(
    HEATING_SETPOINT_K - KELVIN_TO_CELSIUS,
    color="red",
    ls=":",
    lw=1,
    label="Heating SP",
)
ax.set_ylabel("Temp [°C]", fontsize=11)
ax.set_title("VAV Zone Supply Temperature (after reheat coil)", fontsize=13)
ax.legend(fontsize=15, ncol=3)
ax.grid(True, alpha=0.3)
ax.tick_params(axis="both", labelsize=10)

# Row 5: Heat injected to zone (q_zone)
ax = axes_h[5]
for zid in ZONE_IDS:
  ax.plot(
      TS_H,
      q_zone_h[zid],
      color=ZONE_COLORS[zid],
      lw=1.5,
      drawstyle="steps-post",
      label=f"{zid} q_zone",
  )
ax.axhline(0, color="gray", lw=0.8, ls="--")
ax.set_ylabel("Heat [kW]", fontsize=11)
ax.set_title("VAV Heat Injected to Zone (+heating / −cooling)", fontsize=13)
ax.legend(fontsize=15, ncol=2)
ax.grid(True, alpha=0.3)
ax.tick_params(axis="both", labelsize=10)

# Row 6: AHU supply air temperature
ax = axes_h[6]
ax.plot(TS_H, ahu_supply_h, color="teal", lw=1.5, label="AHU supply air")
ax.axhline(
    AHU_SUPPLY_AIR_SP_HEATING_K - KELVIN_TO_CELSIUS,
    color="red",
    ls=":",
    lw=1,
    label=f"AHU supply-air SP ({AHU_SUPPLY_AIR_SP_HEATING_K - KELVIN_TO_CELSIUS:.0f} °C)",
)
ax.set_ylabel("Temp [°C]", fontsize=11)
ax.set_title("Air Handler – Supply Air Temperature", fontsize=13)
ax.legend(fontsize=15)
ax.grid(True, alpha=0.3)
ax.tick_params(axis="both", labelsize=10)

# Row 7: AHU air flow rate
ax = axes_h[7]
ax.plot(
    TS_H,
    ahu_flow_h,
    color="teal",
    lw=1.5,
    drawstyle="steps-post",
    label="AHU air flow [m³/s]",
)
ax.set_ylabel("Flow [m³/s]", fontsize=11)
ax.set_title("Air Handler – Total Air Flow Rate", fontsize=13)
ax.legend(fontsize=15)
ax.grid(True, alpha=0.3)
ax.tick_params(axis="both", labelsize=10)

# Row 8: Boiler supply water temperature
ax = axes_h[8]
ax.plot(
    TS_H, boiler_sw_h, color="firebrick", lw=1.5, label="Boiler supply water"
)
ax.axhline(
    BOILER_SUPPLY_WATER_SP_K - KELVIN_TO_CELSIUS,
    color="red",
    ls="--",
    lw=1,
    label=f"Supply SP ({BOILER_SUPPLY_WATER_SP_K - KELVIN_TO_CELSIUS:.0f} °C)",
)
ax.set_ylabel("Temp [°C]", fontsize=11)
ax.set_title("Boiler – Supply Water Temperature Sensor", fontsize=13)
ax.legend(fontsize=15)
ax.grid(True, alpha=0.3)
ax.tick_params(axis="both", labelsize=10)

# Row 9: Boiler return water temperature
ax = axes_h[9]
ax.plot(TS_H, boiler_rw_h, color="coral", lw=1.5, label="Boiler return water")
ax.set_ylabel("Temp [°C]", fontsize=11)
ax.set_title("Boiler – Return Water Temperature Sensor", fontsize=13)
ax.legend(fontsize=15)
ax.grid(True, alpha=0.3)
ax.tick_params(axis="both", labelsize=10)

# Row 10: HVAC energy rates
ax = axes_h[10]
ax.plot(TS_H, boiler_gas_h, color="firebrick", lw=1.5, label="Boiler gas [kW]")
ax.plot(
    TS_H, fan_elec_h, color="royalblue", lw=1.5, label="Fan electricity [kW]"
)
ax.axhline(0, color="gray", lw=0.8, ls="--")
ax.set_ylabel("Power [kW]", fontsize=11)
ax.set_title("HVAC Energy Rates", fontsize=13)
ax.legend(fontsize=15, ncol=2)
ax.grid(True, alpha=0.3)
ax.tick_params(axis="both", labelsize=10)

# Row 11: Outdoor air temperature
ax = axes_h[11]
ax.plot(TS_H, outdoor_h, color="navy", lw=1.2, label="Outdoor temp")
ax.set_ylabel("Temp [°C]", fontsize=11)
ax.set_xlabel("Timestamp", fontsize=11)
ax.set_title("Outdoor Air Temperature", fontsize=13)
ax.legend(fontsize=15)
ax.grid(True, alpha=0.3)
ax.tick_params(axis="both", labelsize=10)

plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()
print("Heating debug figure complete.")

## Section 8 – TFSimulator: Cooling Scenario (3-day)

Runs a 3-day summer simulation (Jul 15–17) starting from a uniform building
temperature of **310 K (36.85 °C)**. The AHU supplies conditioned air at its
cooling setpoint (26.85 °C), but because this is **above** the comfort upper
bound (23.85 °C), the thermostat cannot engage active `COOL` mode. It
remains in `PASSIVE_COOL` throughout the run, meaning no active heat removal
occurs — any temperature reduction comes solely from envelope heat loss to
the outdoor air.

The same signals as Section 6 are recorded at each step and plotted in
Section 9.

In [ ]:
INITIAL_TEMP_COOLING = 310.0  # K: 36.85 °C hot start.
START_TS_SUMMER = pd.Timestamp("2024-07-15 00:00:00")

bld_cool = create_building(INITIAL_TEMP_COOLING)
# PORT (2026-08-17): cooling uses the cold supply-air setpoint; see cell above.
hvac_cool = create_hvac(
    sorted(bld_cool.get_zone_average_temps()), AHU_SUPPLY_AIR_SP_COOLING_K
)

sim_cool = tf_simulator_py.TFSimulator(
    building=bld_cool,
    hvac=hvac_cool,
    weather_controller=wc_summer,
    time_step_sec=TIME_STEP_SEC,
    convergence_threshold=0.1,
    iteration_limit=100,
    iteration_warning=50,
    start_timestamp=START_TS_SUMMER,
)

print(
    "TFSimulator COOLING init. Mean zone T = "
    f"{bld_cool.temp.mean() - KELVIN_TO_CELSIUS:.2f} °C"
)

# Per-step data storage (mirrors the heating scenario arrays).
ts_cool = []
zone_temp_c = {z: [] for z in ZONE_IDS}
h_sp_c_list, c_sp_c_list, mode_c = [], [], []
damper_c = {z: [] for z in ZONE_IDS}
reheat_v_c = {z: [] for z in ZONE_IDS}
zone_supply_temp_c = {z: [] for z in ZONE_IDS}
q_zone_c = {z: [] for z in ZONE_IDS}
ahu_supply_c = []
ahu_flow_c = []
boiler_sw_c = []
boiler_rw_c = []
boiler_gas_c = []
fan_elec_c = []
outdoor_c = []

for step in range(N_STEPS):
  ts = sim_cool.current_timestamp
  # FIX (2026-07-28): set damper/valve for THIS step before logging.
  sim_cool.setup_step_sim()
  outdoor_k = wc_summer.get_current_temp(ts)
  # Use the mean grid temperature as a proxy for the AHU return-air sensor.
  rec_temp = bld_cool.temp.mean()
  ahu_supply_k = hvac_cool.air_handler.get_supply_air_temp(rec_temp, outdoor_k)

  ts_cool.append(ts)
  sched = zone_vav(hvac_cool, REF_ZONE).thermostat.get_setpoint_schedule()
  h_sp, c_sp = sched.get_temperature_window(ts)
  h_sp_c_list.append(h_sp)
  c_sp_c_list.append(c_sp)
  # pylint: disable=protected-access
  mode_c.append(zone_vav(hvac_cool, REF_ZONE).thermostat._current_mode.value)
  # pylint: enable=protected-access

  zone_avgs = bld_cool.get_zone_average_temps()
  for zid in ZONE_IDS:
    zone_temp_c[zid].append(zone_avgs[zid] - KELVIN_TO_CELSIUS)
    damper_c[zid].append(zone_vav(hvac_cool, zid).damper_setting)
    reheat_v_c[zid].append(zone_vav(hvac_cool, zid).reheat_valve_setting)
    zst_k = zone_vav(hvac_cool, zid).compute_zone_supply_temp(
        ahu_supply_k, hvac_cool.hot_water_system.supply_water_temperature_setpoint
    )
    zone_supply_temp_c[zid].append(zst_k - KELVIN_TO_CELSIUS)
    q_w = zone_vav(hvac_cool, zid).compute_energy_applied_to_zone(
        zone_avgs[zid], ahu_supply_k, hvac_cool.hot_water_system.supply_water_temperature_setpoint
    )
    q_zone_c[zid].append(q_w / 1000)

  ahu_supply_c.append(ahu_supply_k - KELVIN_TO_CELSIUS)
  ahu_flow_c.append(hvac_cool.air_handler.air_flow_rate)

  rw_k = hvac_cool.hot_water_system.return_water_temperature_sensor
  boiler_sw_c.append(
      hvac_cool.hot_water_system.supply_water_temperature_sensor - KELVIN_TO_CELSIUS
  )
  boiler_rw_c.append(rw_k - KELVIN_TO_CELSIUS)
  boiler_gas_c.append(
      hvac_cool.hot_water_system.compute_thermal_energy_rate(rw_k, outdoor_k) / 1000
  )
  fan_elec_c.append(
      (
          hvac_cool.air_handler.compute_intake_fan_energy_rate()
          + hvac_cool.air_handler.compute_exhaust_fan_energy_rate()
      )
      / 1000
  )
  outdoor_c.append(outdoor_k - KELVIN_TO_CELSIUS)

  sim_cool.execute_step_sim()

  if (step + 1) % 288 == 0:
    avg_t = sum(bld_cool.get_zone_average_temps().values()) / len(ZONE_IDS)
    print(
        f'  Day {(step+1)//288} | {ts.strftime("%m-%d %H:%M")}'
        f" | mean T={avg_t - KELVIN_TO_CELSIUS:.2f} °C"
        f" | outdoor={outdoor_k - KELVIN_TO_CELSIUS:.2f} °C"
        f" | mode={mode_c[-1]}"
    )

print("Cooling simulation complete.")

## Section 9 – Cooling Scenario: 12-Row Debug Visualisation

Same 12-panel layout as Section 7 applied to the summer run. Row
definitions are identical; see Section 7 for the panel legend.

Key differences to observe versus the heating scenario:

- **Row 1 (mode)**: stays at `PASSIVE_COOL` (3) for the entire run — the
  thermostat never triggers active cooling because the AHU supply temperature
  (26.85 °C) exceeds the comfort cooling setpoint (23.85 °C).
- **Row 3 (reheat valve)**: fully closed throughout — the boiler is idle.
- **Row 5 (q_zone)**: near zero — no meaningful thermal exchange with the zone.
- **Row 0 (zone temp)**: barely moves despite 3 days of simulation, confirming
  that passive envelope losses alone cannot overcome the 13 °C initial
  overshoot.

In [ ]:
TS_C = ts_cool
fig_c, axes_c = plt.subplots(N_ROWS, 1, figsize=(18, N_ROWS * 3), sharex=True)
fig_c.suptitle(
    "Cooling Scenario (Summer) – 3-Day Debug\n"
    f"(VAV air={VAV_MAX_AIR_FLOW} m³/s, AHU supply-air SP="
    f"{AHU_SUPPLY_AIR_SP_COOLING_K - KELVIN_TO_CELSIUS:.0f} °C)",
    fontsize=13,
)

# Row 0: Zone temperature vs. setpoints.
ax = axes_c[0]
for zid in ZONE_IDS:
  ax.plot(TS_C, zone_temp_c[zid], color=ZONE_COLORS[zid], lw=1.5, label=zid)
ax.plot(
    TS_C,
    [t - KELVIN_TO_CELSIUS for t in h_sp_c_list],
    color="red",
    lw=1.2,
    ls="--",
    drawstyle="steps-post",
    label="Heating SP",
)
ax.plot(
    TS_C,
    [t - KELVIN_TO_CELSIUS for t in c_sp_c_list],
    color="orange",
    lw=1.2,
    ls="--",
    drawstyle="steps-post",
    label="Cooling SP",
)
ax.set_ylabel("Temp [°C]")
ax.set_title("Zone Air Temperature vs. Setpoints")
ax.legend(fontsize=8, ncol=4, loc="upper right")
ax.grid(True, alpha=0.3)

# Row 1: Thermostat mode.
ax = axes_c[1]
ax.plot(
    TS_C,
    mode_c,
    color="purple",
    lw=1.5,
    drawstyle="steps-post",
    label="Mode (0=OFF,1=HEAT,2=COOL,3=PASSIVE_COOL)",
)
ax.set_ylabel("Mode")
ax.set_yticks([0, 1, 2, 3])
ax.set_yticklabels(["OFF", "HEAT", "COOL", "P.COOL"])
ax.set_title("Thermostat Mode")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Row 2: VAV damper.
ax = axes_c[2]
for zid in ZONE_IDS:
  ax.plot(
      TS_C,
      damper_c[zid],
      color=ZONE_COLORS[zid],
      lw=1.2,
      drawstyle="steps-post",
      label=f"{zid} damper",
  )
ax.set_ylabel("Damper [0-1]")
ax.set_ylim(-0.05, 1.05)
ax.set_title("VAV Damper Position")
ax.legend(fontsize=8, ncol=2)
ax.grid(True, alpha=0.3)

# Row 3: VAV reheat valve.
ax = axes_c[3]
for zid in ZONE_IDS:
  ax.plot(
      TS_C,
      reheat_v_c[zid],
      color=ZONE_COLORS[zid],
      lw=1.2,
      drawstyle="steps-post",
      label=f"{zid} reheat valve",
  )
ax.set_ylabel("Reheat Valve [0-1]")
ax.set_ylim(-0.05, 1.05)
ax.set_title("VAV Reheat Valve Position")
ax.legend(fontsize=8, ncol=2)
ax.grid(True, alpha=0.3)

# Row 4: Zone supply temperature.
ax = axes_c[4]
for zid in ZONE_IDS:
  ax.plot(
      TS_C,
      zone_supply_temp_c[zid],
      color=ZONE_COLORS[zid],
      lw=1.5,
      label=f"{zid} zone supply",
  )
ax.axhline(
    COOLING_SETPOINT_K - KELVIN_TO_CELSIUS,
    color="orange",
    ls=":",
    lw=1,
    label="Cooling SP",
)
ax.set_ylabel("Temp [°C]")
ax.set_title("VAV Zone Supply Temperature (after reheat coil)")
ax.legend(fontsize=8, ncol=3)
ax.grid(True, alpha=0.3)

# Row 5: Heat injected to zone (negative = cooling effect).
ax = axes_c[5]
for zid in ZONE_IDS:
  ax.plot(
      TS_C,
      q_zone_c[zid],
      color=ZONE_COLORS[zid],
      lw=1.5,
      drawstyle="steps-post",
      label=f"{zid} q_zone",
  )
ax.axhline(0, color="gray", lw=0.8, ls="--")
ax.set_ylabel("Heat [kW]")
ax.set_title("VAV Heat Injected to Zone (negative = cooling)")
ax.legend(fontsize=8, ncol=2)
ax.grid(True, alpha=0.3)

# Row 6: AHU supply air temperature.
ax = axes_c[6]
ax.plot(TS_C, ahu_supply_c, color="teal", lw=1.5, label="AHU supply air")
ax.axhline(
    AHU_SUPPLY_AIR_SP_COOLING_K - KELVIN_TO_CELSIUS,
    color="orange",
    ls=":",
    lw=1,
    label=f"AHU supply-air SP ({AHU_SUPPLY_AIR_SP_COOLING_K - KELVIN_TO_CELSIUS:.0f} °C)",
)
ax.set_ylabel("Temp [°C]")
ax.set_title("Air Handler – Supply Air Temperature")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Row 7: AHU air flow rate.
ax = axes_c[7]
ax.plot(
    TS_C,
    ahu_flow_c,
    color="teal",
    lw=1.5,
    drawstyle="steps-post",
    label="AHU air flow [m³/s]",
)
ax.set_ylabel("Flow [m³/s]")
ax.set_title("Air Handler – Total Air Flow Rate")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Row 8: Boiler supply water temperature.
ax = axes_c[8]
ax.plot(
    TS_C, boiler_sw_c, color="firebrick", lw=1.5, label="Boiler supply water"
)
ax.axhline(
    BOILER_SUPPLY_WATER_SP_K - KELVIN_TO_CELSIUS,
    color="red",
    ls="--",
    lw=1,
    label=f"Supply SP ({BOILER_SUPPLY_WATER_SP_K - KELVIN_TO_CELSIUS:.0f} °C)",
)
ax.set_ylabel("Temp [°C]")
ax.set_title("Boiler – Supply Water Temperature Sensor")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Row 9: Boiler return water temperature.
ax = axes_c[9]
ax.plot(TS_C, boiler_rw_c, color="coral", lw=1.5, label="Boiler return water")
ax.set_ylabel("Temp [°C]")
ax.set_title("Boiler – Return Water Temperature Sensor")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Row 10: HVAC energy rates.
ax = axes_c[10]
ax.plot(TS_C, boiler_gas_c, color="firebrick", lw=1.5, label="Boiler gas [kW]")
ax.plot(
    TS_C, fan_elec_c, color="royalblue", lw=1.5, label="Fan electricity [kW]"
)
ax.axhline(0, color="gray", lw=0.8, ls="--")
ax.set_ylabel("Power [kW]")
ax.set_title("HVAC Energy Rates")
ax.legend(fontsize=8, ncol=2)
ax.grid(True, alpha=0.3)

# Row 11: Outdoor air temperature.
ax = axes_c[11]
ax.plot(TS_C, outdoor_c, color="saddlebrown", lw=1.2, label="Outdoor temp")
ax.set_ylabel("Temp [°C]")
ax.set_xlabel("Timestamp")
ax.set_title("Outdoor Air Temperature")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()
print("Cooling debug figure complete.")

## Section 10 – Scenario Comparison

Overlays both scenarios on a common time axis (hours from simulation start)
so that the qualitative differences are immediately visible side-by-side.
Solid lines = heating scenario; dashed lines = cooling scenario.
Both zones share the same colour (room_1 = blue, room_2 = orange) across rows.

| Row | Signal | What to observe |
|-----|--------|-----------------|
| 0 | Zone air temperature vs. comfort setpoints | Heating crosses into comfort band; cooling barely descends |
| 1 | VAV zone supply temperature | Heating shows elevated supply from reheat coil; cooling supply is constant at AHU setpoint |
| 2 | Heat injected to zone [kW] | Heating delivers sustained positive flux; cooling q_zone ≈ 0 |
| 3 | HVAC energy rates [kW] | Heating shows large boiler gas consumption; cooling shows fan electricity only |

In [ ]:
def to_hours(ts_list, t0):
  """Converts a list of timestamps to elapsed hours from a reference time.

  Args:
    ts_list: Sequence of pandas Timestamps.
    t0: Reference pandas Timestamp (time zero).

  Returns:
    List of floats representing hours elapsed since t0.
  """
  return [(t - t0).total_seconds() / 3600 for t in ts_list]


hours_h = to_hours(ts_heat, ts_heat[0])
hours_c = to_hours(ts_cool, ts_cool[0])

fig_cmp, axes_cmp = plt.subplots(4, 1, figsize=(16, 16), sharex=True)
fig_cmp.suptitle(
    f"Heating vs. Cooling – Overlaid (0–{N_DAYS*24} h)", fontsize=13
)

# Row 0: Zone air temperatures – solid = heating, dashed = cooling.
ax = axes_cmp[0]
for zid in ZONE_IDS:
  ax.plot(
      hours_h,
      zone_temp_h[zid],
      color=ZONE_COLORS[zid],
      lw=1.5,
      ls="-",
      label=f"{zid} (heat)",
  )
  ax.plot(
      hours_c,
      zone_temp_c[zid],
      color=ZONE_COLORS[zid],
      lw=1.5,
      ls="--",
      label=f"{zid} (cool)",
  )
ax.axhline(h_sp_c, color="red", ls=":", lw=1, label=f"HeatSP {h_sp_c:.0f} °C")
ax.axhline(
    c_sp_c, color="orange", ls=":", lw=1, label=f"CoolSP {c_sp_c:.0f} °C"
)
ax.set_ylabel("Temp [°C]")
ax.set_title("Zone Air Temperatures")
ax.legend(fontsize=8, ncol=3)
ax.grid(True, alpha=0.3)

# Row 1: VAV zone supply temperature.
ax = axes_cmp[1]
for zid in ZONE_IDS:
  ax.plot(
      hours_h,
      zone_supply_temp_h[zid],
      color=ZONE_COLORS[zid],
      lw=1.5,
      ls="-",
      label=f"{zid} supply (heat)",
  )
  ax.plot(
      hours_c,
      zone_supply_temp_c[zid],
      color=ZONE_COLORS[zid],
      lw=1.5,
      ls="--",
      label=f"{zid} supply (cool)",
  )
ax.set_ylabel("Temp [°C]")
ax.set_title("VAV Zone Supply Temperature")
ax.legend(fontsize=8, ncol=2)
ax.grid(True, alpha=0.3)

# Row 2: Heat injected to zone.
ax = axes_cmp[2]
for zid in ZONE_IDS:
  ax.plot(
      hours_h,
      q_zone_h[zid],
      color=ZONE_COLORS[zid],
      lw=1.5,
      ls="-",
      drawstyle="steps-post",
      label=f"{zid} q (heat)",
  )
  ax.plot(
      hours_c,
      q_zone_c[zid],
      color=ZONE_COLORS[zid],
      lw=1.5,
      ls="--",
      drawstyle="steps-post",
      label=f"{zid} q (cool)",
  )
ax.axhline(0, color="gray", lw=0.8, ls="--")
ax.set_ylabel("Heat [kW]")
ax.set_title("Heat Injected to Zone")
ax.legend(fontsize=8, ncol=2)
ax.grid(True, alpha=0.3)

# Row 3: HVAC energy rates.
ax = axes_cmp[3]
ax.plot(
    hours_h,
    boiler_gas_h,
    color="firebrick",
    lw=1.5,
    label="Boiler gas (heat) [kW]",
)
ax.plot(
    hours_c,
    boiler_gas_c,
    color="firebrick",
    lw=1.5,
    ls="--",
    label="Boiler gas (cool) [kW]",
)
ax.plot(
    hours_h,
    fan_elec_h,
    color="royalblue",
    lw=1.5,
    label="Fan elec (heat) [kW]",
)
ax.plot(
    hours_c,
    fan_elec_c,
    color="royalblue",
    lw=1.5,
    ls="--",
    label="Fan elec (cool) [kW]",
)
ax.set_ylabel("Power [kW]")
ax.set_xlabel("Hours from start")
ax.set_title("HVAC Energy Rates")
ax.legend(fontsize=8, ncol=2)
ax.grid(True, alpha=0.3)

plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()
print("Comparison figure complete.")

## Section 11 – Summary

Aggregates the 3-day simulation into a single comparison table with the
following metrics per scenario:

| Metric | Description |
|--------|-------------|
| Min / Max zone temp [°C] | Extreme temperatures observed across both zones over the full run |
| % Time in comfort | Fraction of (step, zone) samples where the zone temperature fell within the active comfort band |
| Boiler gas [kWh] | Total thermal energy consumed by the boiler (heating only) |
| Fan elec [kWh] | Total electrical energy consumed by the AHU fans |
| Total HVAC [kWh] | Combined energy (boiler gas + fan electricity) |

**Expected results**

- **Winter heating**: comfort fraction ~59 %, dominated by boiler gas (~695 kWh) with minor fan contribution.
- **Summer cooling**: comfort fraction 0 % — the AHU supply setpoint (26.85 °C) exceeds the comfort upper bound (23.85 °C), so active cooling never triggers. Total energy is fan electricity only (~64 kWh).

In [ ]:
DT_H = TIME_STEP_SEC / 3600  # Hours per simulation step.

# ---------------------------------------------------------------------------
# FIX (2026-07-28): comfort is now reported separately for OCCUPIED and
# UNOCCUPIED hours.
#
# The previous single "% Time in comfort" number scored every step against the
# *scheduled* band, which relaxes to the eco window (14.85-26.85 °C) outside
# 09:00-18:00. Since eco hours are 15/24 = 62.5 % of the run, a building that
# never once reached its occupied setpoint still scored 59.4 % -- essentially
# all of it unoccupied setback time. That reads as partial success while the
# building actually fails whenever anyone is in it.
# ---------------------------------------------------------------------------
OCCUPIED_START_HOUR = 9
OCCUPIED_END_HOUR = 18


def scenario_summary(
    scenario_label,
    zone_temps_c,
    h_sp_k_list,
    c_sp_k_list,
    gas_list,
    fan_list,
    ts_list,
):
  """Computes aggregate performance metrics for one simulation scenario.

  Args:
    scenario_label: Human-readable name used as the DataFrame index key.
    zone_temps_c: Dict mapping zone id to list of temperatures in °C,
      one entry per simulation step.
    h_sp_k_list: List of heating setpoint values in Kelvin, one per step.
    c_sp_k_list: List of cooling setpoint values in Kelvin, one per step.
    gas_list: List of boiler gas power values in kW, one per step.
    fan_list: List of fan electrical power values in kW, one per step.
    ts_list: List of pandas Timestamps, one per step, used to split occupied
      from unoccupied hours.

  Returns:
    Dict of summary metrics, including comfort split by occupancy and a
    plausibility check of total energy against the envelope design load.
  """
  all_temps = []
  for zone_key in ZONE_IDS:
    all_temps.extend(zone_temps_c[zone_key])
  total_gas_kwh = sum(gas_list) * DT_H
  total_fan_kwh = sum(fan_list) * DT_H

  occ_hits, occ_total, uno_hits, uno_total = 0, 0, 0, 0
  for i, ts_i in enumerate(ts_list):
    occupied = OCCUPIED_START_HOUR <= ts_i.hour < OCCUPIED_END_HOUR
    lo = h_sp_k_list[i] - KELVIN_TO_CELSIUS
    hi = c_sp_k_list[i] - KELVIN_TO_CELSIUS
    for zid in ZONE_IDS:
      in_band = lo <= zone_temps_c[zid][i] <= hi
      if occupied:
        occ_total += 1
        occ_hits += in_band
      else:
        uno_total += 1
        uno_hits += in_band

  def pct(hits, total):
    return round(hits / total * 100, 1) if total else float("nan")

  # Plausibility check: average HVAC power against the envelope design load
  # computed in Section 2. A ratio far above 1 means the model is delivering
  # far more energy than the building can physically lose.
  hours = len(ts_list) * DT_H
  avg_kw = (total_gas_kwh + total_fan_kwh) / hours if hours else float("nan")
  design_kw = DESIGN_HEATING_W / 1000.0

  return {
      "Scenario": scenario_label,
      "Min zone temp [C]": round(min(all_temps), 2),
      "Max zone temp [C]": round(max(all_temps), 2),
      "% Comfort (occupied)": pct(occ_hits, occ_total),
      "% Comfort (unoccupied)": pct(uno_hits, uno_total),
      "Boiler gas [kWh]": round(total_gas_kwh, 1),
      "Fan elec [kWh]": round(total_fan_kwh, 2),
      "Total HVAC [kWh]": round(total_gas_kwh + total_fan_kwh, 1),
      "Avg HVAC [kW]": round(avg_kw, 2),
      "Avg / design capacity": round(avg_kw / design_kw, 2),
  }


smry_h = scenario_summary(
    "Winter Heating",
    zone_temp_h,
    h_sp_h,
    c_sp_h,
    boiler_gas_h,
    fan_elec_h,
    ts_heat,
)
smry_c = scenario_summary(
    "Summer Cooling",
    zone_temp_c,
    h_sp_c_list,
    c_sp_c_list,
    boiler_gas_c,
    fan_elec_c,
    ts_cool,
)

smry_df = pd.DataFrame([smry_h, smry_c]).set_index("Scenario")
print("=" * 100)
print("SIMULATION SUMMARY")
print("=" * 100)
print(smry_df.to_string())
print()
print(f"Envelope steady-state loss (Section 2): 262 W;"
      f" design capacity {DESIGN_HEATING_W/1000:.2f} kW")
print("'Avg / design capacity' should be well below 1 for a 3-day run that")
print("spends most of its time at setback. Values above 1 indicate the model")
print("is delivering more energy than the envelope can plausibly lose.")
print()
print("Notebook complete.")
